In [ ]:
!pip install -q ultralytics
!pip install -q fastapi uvicorn nest-asyncio pyngrok python-multipart
!pip install -q opencv-python-headless
!pip install -q torch

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("PASTE YOUR NGROK AUTH TOKEN HERE")

In [ ]:
from ultralytics import YOLO

from fastapi import FastAPI, UploadFile, File
import uvicorn
import nest_asyncio
from pyngrok import ngrok

import cv2
import shutil
import os
import torch

In [ ]:
%%writefile app.py

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
from ultralytics import YOLO
import torch
import cv2
import numpy as np

app = FastAPI(
    title="YOLOv8 Emotion Detection API",
)

# -----------------------------
# Load Model
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    model = YOLO("/content/best.pt")
    model.to(device)
    model_loaded = True
    print(f"✅ Model loaded successfully on {device}")
except Exception as e:
    model_loaded = False
    model = None
    print(f"❌ Failed to load model: {e}")


# -----------------------------
# Root Endpoint
# -----------------------------
@app.get("/")
def home():
    return {
        "message": "YOLOv8 REST API is running",
        "device": device,
        "model_loaded": model_loaded
    }


# -----------------------------
# Health Check
# -----------------------------
@app.get("/health")
def health():
    if not model_loaded:
        raise HTTPException(
            status_code=503,
            detail="Model failed to load."
        )

    return {
        "status": "healthy",
        "device": device
    }


# -----------------------------
# Prediction Endpoint
# -----------------------------
@app.post("/predict")
async def predict(file: UploadFile = File(...)):

    # Check model availability
    if not model_loaded:
        raise HTTPException(
            status_code=503,
            detail="Model is unavailable."
        )

    # Check file exists
    if file.filename == "":
        raise HTTPException(
            status_code=400,
            detail="No file uploaded."
        )

    # Check file type
    allowed_types = [
        "image/jpeg",
        "image/png",
        "image/jpg",
        "image/webp",
        "image/bmp"
    ]

    if file.content_type not in allowed_types:
        raise HTTPException(
            status_code=415,
            detail=f"Unsupported file type '{file.content_type}'. "
                   f"Supported types: jpg, jpeg, png, webp, bmp."
        )

    try:
        contents = await file.read()

        if len(contents) == 0:
            raise HTTPException(
                status_code=400,
                detail="Uploaded file is empty."
            )

        img = cv2.imdecode(
            np.frombuffer(contents, np.uint8),
            cv2.IMREAD_COLOR
        )

        if img is None:
            raise HTTPException(
                status_code=400,
                detail="Invalid or corrupted image."
            )

        # Run inference
        results = model(img)

        detections = []

        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(float, box.xyxy[0])
                cls = int(box.cls[0])

                detections.append({
                    "class": model.names[cls],
                    "confidence": round(float(box.conf[0]), 4),
                    "bbox": [
                        round(x1, 2),
                        round(y1, 2),
                        round(x2, 2),
                        round(y2, 2)
                    ]
                })

        return {
            "success": True,
            "device": device,
            "num_detections": len(detections),
            "detections": detections
        }

    except HTTPException:
        raise

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Inference failed: {str(e)}"
        )


# -----------------------------
# Global Exception Handler
# -----------------------------
@app.exception_handler(Exception)
async def global_exception_handler(request, exc):
    return JSONResponse(
        status_code=500,
        content={
            "success": False,
            "error": "Internal Server Error",
            "detail": str(exc)
        }
    )

Writing app.py


In [ ]:
import os
import subprocess
import time

# Kill any process using port 8000
if 'server' in locals() and server.poll() is None:
  os.system("fuser -k 8000/tcp")
  time.sleep(2)
  print("Existing Server Stopped!")

# Start server
server = subprocess.Popen([
    "uvicorn",
    "app:app",
    "--host", "0.0.0.0",
    "--port", "8000"
])

time.sleep(5)
print("Server started!")

Server started!


In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)
print(public_url)

NgrokTunnel: "https://perkiness-aftermost-diffusive.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
# Test if the server is running.
!netstat -tlnp | grep 8000

tcp        0      0 0.0.0.0:8000            0.0.0.0:*               LISTEN      21692/python3       


Kill the server

In [ ]:
# Kill any process using port 8000
if 'server' in locals() and server.poll() is None:
  os.system("fuser -k 8000/tcp")